# RAG MVP (recetas)

Este cuaderno orquesta el pipeline usando los modulos en src/.

In [4]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/xabierlosa/ProyectoPLN.git"
ROOT = Path("/content/ProyectoPLN")

if not (ROOT / ".git").exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT))
print(f"Repositorio listo en {ROOT}")

Repositorio listo en /content/ProyectoPLN


In [2]:
# En Colab, instala un stack compatible con Python 3.12 y valida imports en un proceso limpio.
import os
import subprocess
import sys
from pathlib import Path

restart_marker = Path("/content/.proyectopln_deps_ready_v4")

check_code = """
import numpy
import scipy
import sklearn
import torch
import transformers
import sentence_transformers
import faiss
print('OK', numpy.__version__, scipy.__version__, sklearn.__version__, torch.__version__, transformers.__version__, sentence_transformers.__version__)
""".strip()

def run_checked(cmd, label):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"[{label}] fallo con codigo {result.returncode}")
        if result.stdout:
            print("--- STDOUT ---")
            print(result.stdout[-3500:])
        if result.stderr:
            print("--- STDERR ---")
            print(result.stderr[-3500:])
        return False
    return True

profiles = [
    {
        "name": "py312-faiss190-numpy2",
        "packages": [
            "numpy==2.0.2",
            "scipy==1.14.1",
            "scikit-learn==1.5.2",
            "torch==2.4.1",
            "transformers==4.44.2",
            "sentence-transformers==3.1.1",
            "accelerate==0.34.2",
            "safetensors==0.4.5",
            "faiss-cpu==1.9.0.post1",
        ],
    },
    {
        "name": "py312-faiss190-numpy1",
        "packages": [
            "numpy==1.26.4",
            "scipy==1.13.1",
            "scikit-learn==1.5.1",
            "torch==2.4.1",
            "transformers==4.44.2",
            "sentence-transformers==3.1.1",
            "accelerate==0.34.2",
            "safetensors==0.4.5",
            "faiss-cpu==1.9.0.post1",
        ],
    },
]

if not restart_marker.exists():
    run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "faiss-cpu", "sentence-transformers", "scikit-learn", "scipy", "numpy"], "cleanup")

    validated = False
    for profile in profiles:
        ok = run_checked(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "--force-reinstall",
                "--no-cache-dir",
                *profile["packages"],
            ],
            f"install-{profile['name']}",
        )
        if not ok:
            continue

        ok = run_checked([sys.executable, "-c", check_code], f"check-{profile['name']}")
        if ok:
            print(f"Perfil valido: {profile['name']}")
            validated = True
            break

    if not validated:
        raise RuntimeError("No se pudo validar un stack binario compatible en Colab. Revisa STDERR arriba.")

    restart_marker.write_text("ok")

    if "google.colab" in sys.modules:
        print("Reiniciando el runtime de Colab para cargar los binarios nuevos...")
        os.kill(os.getpid(), 9)
else:
    print("Dependencias validadas. Continua con la siguiente celda.")

Dependencias validadas. Continua con la siguiente celda.


In [ ]:
from pathlib import Path
import sys

ROOT = Path("/content/ProyectoPLN")
sys.path.insert(0, str(ROOT))

import torch

from src.chunking import chunk_documents
from src.data_loader import build_documents, load_recipes
from src.embeddings import embed_passages, load_embedder
from src.faiss_index import build_index, save_docs, save_index

DATA_PATH = ROOT / "ChefGPT_Dataset_Random_Sample.json"
OUT_DIR = ROOT / "artifacts"

recipes = load_recipes(DATA_PATH)
docs = build_documents(recipes)
chunks = chunk_documents(docs, max_words=220, overlap=40)

texts = [doc["text"] for doc in chunks]
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = load_embedder("intfloat/multilingual-e5-base", device=device)
embeddings = embed_passages(embedder, texts, batch_size=32)

index = build_index(embeddings)
save_index(index, OUT_DIR / "index.faiss")
save_docs(chunks, OUT_DIR / "docs.jsonl")

print(f"Index listo: {len(chunks)} chunks")

ModuleNotFoundError: No module named 'src'

In [ ]:
from pathlib import Path
import sys

ROOT = Path("/content/ProyectoPLN")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.embeddings import embed_query
from src.faiss_index import load_docs, load_index
from src.rag_pipeline import generate_answer, load_llm

OUT_DIR = ROOT / "artifacts"

index = load_index(OUT_DIR / "index.faiss")
docs = load_docs(OUT_DIR / "docs.jsonl")

query = "Tengo pollo y arroz. Que puedo cocinar rapido?"
query_vec = embed_query(embedder, query).astype("float32")
scores, indices = index.search(query_vec.reshape(1, -1), 6)
contexts = [docs[i]["text"] for i in indices[0] if i >= 0]

tokenizer, model = load_llm("Qwen/Qwen2.5-7B-Instruct")
answer = generate_answer(tokenizer, model, query, contexts)
print(answer)